# Generate Reference Answers for ALICE

This notebook generates multiple reference answers for the ALICE dataset. 
It shows the LLM the existing sample solution and several student answers with the highest 
learning performance score as context.

In [6]:
import os
import json
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple
import time
import random

import openai
from tqdm import tqdm

API_KEY = "REDACTED_OPENAI_KEY" # 用户提供的 API Key
client = openai.OpenAI(api_key=API_KEY)

MODEL = "gpt-4o-mini"
ALICE_DIR = Path("alice_data")
SPLIT_FILES = ["train.json", "test_uq.json", "test_ua.json"]
LABEL_FIELD = "learning_performance"
MAX_PER_QUESTION = 10
MAX_STUDENT_EXAMPLES = 10
SEED = 42
random.seed(SEED)

print("Ready. Set OPENAI_API_KEY to enable LLM calls.")


Ready. Set OPENAI_API_KEY to enable LLM calls.


In [3]:
def load_split_records(alice_dir: Path, split_files: Iterable[str]) -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    for name in split_files:
        path = alice_dir / name
        if not path.exists():
            print(f"Missing split: {path}")
            continue
        data = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(data, list):
            print(f"Unexpected format in {path}; expected a list.")
            continue
        records.extend(data)
    return records


def parse_label_value(value: Any) -> Optional[float]:
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return float(value)
    s = str(value).strip()
    if not s or s.lower() == "nan":
        return None
    try:
        return float(int(s))
    except Exception:
        try:
            return float(s)
        except Exception:
            return None


def extract_label(row: Dict[str, Any], field: str) -> Optional[float]:
    data = row.get(field)
    if isinstance(data, dict):
        values = [parse_label_value(v) for v in data.values()]
        values = [v for v in values if v is not None]
        return max(values) if values else None
    return parse_label_value(data)


def collect_top_answers(
    records: List[Dict[str, Any]],
    question_id: str,
    label_field: str,
    max_examples: int,
) -> Tuple[List[str], Optional[float]]:
    max_score: Optional[float] = None
    answers: List[str] = []
    for row in records:
        if str(row.get("question_id")) != str(question_id):
            continue
        score = extract_label(row, label_field)
        if score is None:
            continue
        answer = str(row.get("answer", "")).strip()
        if not answer:
            continue
        if max_score is None or score > max_score:
            max_score = score
            answers = [answer]
        elif score == max_score:
            answers.append(answer)

    seen = set()
    uniq: List[str] = []
    for ans in answers:
        if ans in seen:
            continue
        seen.add(ans)
        uniq.append(ans)
        if len(uniq) >= max_examples:
            break
    return uniq, max_score


## Prompt Template

The prompt includes the dataset's sample solution and a few highest-scoring student answers 
for learning performance. The model should return a JSON object with an "answers" list.

In [4]:
def build_prompt(
    question_text: str,
    sample_solution: Any,
    student_examples: List[str],
    n: int,
) -> str:
    header = (
        f"Generate {n} high-quality, diverse reference answers for the following question. \n"
        f"Each answer should be a valid response that could be used as a scoring reference.\n\n"
    )
    q_block = f"Question:\n{question_text}\n\n"

    sample_block = ""
    if sample_solution:
        if isinstance(sample_solution, list):
            items = [str(s).strip() for s in sample_solution if str(s).strip()]
            sample_block = "Sample solution(s):\n" + "\n".join([f"- {s}" for s in items[:5]]) + "\n\n"
        else:
            sample_block = "Sample solution:\n- " + str(sample_solution).strip()[:1200] + "\n\n"

    examples_block = ""
    if student_examples:
        examples_block = (
            "Highest-scoring student answers (learning performance):\n"
            + "\n".join([f"- {e[:400]}" for e in student_examples])
            + "\n\n"
        )

    footer = (
        "Requirements:\n"
        "- Answers must be correct and aligned with the question.\n"
        "- Use diverse wording and focus points across answers.\n"
        "- Keep answers concise; avoid scoring labels or meta-comments.\n"
        "- Ensure the language matches the question.\n"
        "- Return JSON only: {\"answers\": [\"...\", ...]}.\n"
    )
    return header + q_block + sample_block + examples_block + footer


In [7]:
def call_llm_for_answers(
    prompt: str,
    model: str = MODEL,
    max_tokens: int = 800,
    temperature: float = 0.8,
    retries: int = 3,
) -> List[str]:
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            if not (API_KEY or os.environ.get("OPENAI_API_KEY")):
                raise RuntimeError("No API key found. Set OPENAI_API_KEY.")
            resp = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are a helpful, precise assistant."},
                    {"role": "user", "content": prompt},
                ],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            try:
                text = resp.choices[0].message.content
            except Exception:
                text = resp["choices"][0]["message"]["content"]

            text = text.strip()
            if text.startswith("```"):
                text = text.strip('`')
                text = text.replace("json", "", 1).strip()

            try:
                parsed = json.loads(text)
                if isinstance(parsed, dict) and "answers" in parsed:
                    return parsed["answers"]
                if isinstance(parsed, list):
                    return parsed
            except Exception:
                lines = [l.strip() for l in text.splitlines() if l.strip()]
                answers = []
                for line in lines:
                    import re
                    cleaned = re.sub(r"^\s*\d+[.)\s-]*", "", line)
                    if cleaned:
                        answers.append(cleaned)
                if answers:
                    return answers[:MAX_PER_QUESTION]
                return []

            raise ValueError("Unable to parse model output as answers list")
        except Exception as e:
            last_err = e
            wait = 2 ** attempt
            print(f"LLM call error (attempt {attempt}/{retries}): {e}. Waiting {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"LLM calls failed: {last_err}")


In [8]:
def generate_for_alice(
    alice_dir: Path = ALICE_DIR,
    split_files: Iterable[str] = SPLIT_FILES,
    label_field: str = LABEL_FIELD,
    max_per_question: int = MAX_PER_QUESTION,
    max_examples: int = MAX_STUDENT_EXAMPLES,
    model: str = MODEL,
    dry_run: bool = False,
) -> Path:
    meta_path = alice_dir / "question_meta.json"
    if not meta_path.exists():
        raise FileNotFoundError(f"Missing question_meta.json at {meta_path}")
    qmeta = json.loads(meta_path.read_text(encoding="utf-8"))
    records = load_split_records(alice_dir, split_files)

    results: Dict[str, Dict[str, Any]] = {}
    for qid, info in tqdm(qmeta.items(), desc="Generating ALICE"):
        question_text = info.get("prompt", info.get("question", ""))
        sample_solution = info.get("sample_solution", None)
        examples, max_score = collect_top_answers(
            records, qid, label_field=label_field, max_examples=max_examples
        )

        prompt = build_prompt(
            question_text=question_text,
            sample_solution=sample_solution,
            student_examples=examples,
            n=max_per_question,
        )

        if dry_run or not (API_KEY or os.environ.get("OPENAI_API_KEY")):
            print(f"--- DRY RUN: {qid} (max LP={max_score}) ---")
            print(prompt)
            results[qid] = {
                "question_text": question_text,
                "sample_solution": sample_solution,
                "answers": [],
            }
            continue

        try:
            answers = call_llm_for_answers(prompt, model=model, temperature=0.8)
            uniq: List[str] = []
            for ans in answers:
                if ans not in uniq:
                    uniq.append(ans)
            results[qid] = {
                "question_text": question_text,
                "sample_solution": sample_solution,
                "answers": uniq[:max_per_question],
            }
        except Exception as e:
            print(f"Question {qid} failed: {e}")
            results[qid] = {
                "question_text": question_text,
                "sample_solution": sample_solution,
                "answers": [],
                "error": str(e),
            }

    out_path = alice_dir / "llm_reference_answers.json"
    out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Saved: {out_path} (questions: {len(results)})")
    return out_path


# Set dry_run=True to preview prompts without calling the API.
generate_for_alice(dry_run=False)


Generating ALICE: 100%|██████████| 208/208 [38:35<00:00, 11.13s/it]

Saved: alice_data/llm_reference_answers.json (questions: 208)


PosixPath('alice_data/llm_reference_answers.json')